# Exploratory Data Analysis — Hiver AI Customer Support Agent

This notebook demonstrates the data exploration process used to:
1. Understand the TWCS dataset structure
2. Select the optimal brand (AmazonHelp)
3. Reconstruct conversations from tweet reply trees
4. Analyze intent distribution across the golden evaluation set

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
from pathlib import Path
from collections import Counter

## 1. Load Sample Dataset

In [ ]:
df = pd.read_csv('../data/raw/sample_twcs.csv')
print(f'Total tweets: {len(df)}')
print(f'Columns: {list(df.columns)}')
print(f'\nAuthor IDs (brands + customers):')
print(df['author_id'].value_counts().head(10))
df.head()

## 2. Brand Analysis

In [ ]:
brand_tweets = df[df['author_id'].str.contains('Amazon|apple|Uber|Sprint|Spotify', case=False, na=False)]
print(f'Brand tweets: {len(brand_tweets)}')
print(brand_tweets['author_id'].value_counts())

## 3. Reconstructed Conversations

In [ ]:
conv_path = Path('../data/processed/conversations.json')
if conv_path.exists():
    convs = json.load(open(conv_path))
    print(f'Total conversations: {len(convs)}')
    print(f'Avg messages per conversation: {sum(len(c["messages"]) for c in convs) / len(convs):.1f}')
    print(f'\nExample conversation:')
    for msg in convs[0]['messages'][:4]:
        print(f'  [{msg["author"]}] {msg["text"][:80]}...')
else:
    print('Run scripts/build_conversations.py first.')

## 4. Golden Set Intent Distribution

In [ ]:
golden_path = Path('../data/golden/golden_set.jsonl')
if golden_path.exists():
    records = [json.loads(line) for line in open(golden_path) if line.strip()]
    print(f'Golden set size: {len(records)}')
    
    intent_counts = Counter(r['gold_intent'] for r in records)
    esc_counts = Counter(r['gold_escalation'] for r in records)
    diff_counts = Counter(r['difficulty'] for r in records)
    
    print(f'\nIntent Distribution:')
    for intent, count in sorted(intent_counts.items(), key=lambda x: -x[1]):
        print(f'  {intent:25s} {count:3d} ({count/len(records)*100:.1f}%)')
    
    print(f'\nEscalation Distribution:')
    for dec, count in esc_counts.items():
        print(f'  {dec:15s} {count:3d} ({count/len(records)*100:.1f}%)')
    
    print(f'\nDifficulty Distribution:')
    for diff, count in diff_counts.items():
        print(f'  {diff:10s} {count:3d} ({count/len(records)*100:.1f}%)')
else:
    print('Run scripts/create_golden_set.py first.')

## 5. Evaluation Results Summary

In [ ]:
eval_path = Path('../artifacts/evaluation_results.json')
if eval_path.exists():
    results = json.load(open(eval_path))
    
    rows = []
    for sys_name, data in results.items():
        im = data['intent_metrics']
        em = data['escalation_metrics']
        rows.append({
            'System': sys_name,
            'Intent Accuracy': f"{im['accuracy']*100:.1f}%",
            'Macro F1': f"{im['macro_f1']:.3f}",
            'Escalation F1': f"{em['f1']:.3f}",
            'Unsafe Auto Rate': f"{em['unsafe_auto_handling_rate']*100:.1f}%",
            'Avg Reply Score': data['avg_reply_score']
        })
    
    df_results = pd.DataFrame(rows)
    print('Multi-Baseline Evaluation Results:')
    print(df_results.to_string(index=False))
else:
    print('Run scripts/run_evaluation.py first.')

## 6. Leak Check Verification

In [ ]:
dev = json.load(open('../data/processed/dev_conversations.json')) if Path('../data/processed/dev_conversations.json').exists() else []
val = json.load(open('../data/processed/val_conversations.json')) if Path('../data/processed/val_conversations.json').exists() else []
evl = json.load(open('../data/processed/eval_conversations.json')) if Path('../data/processed/eval_conversations.json').exists() else []

dev_ids = {c['conversation_id'] for c in dev}
val_ids = {c['conversation_id'] for c in val}
evl_ids = {c['conversation_id'] for c in evl}

print(f'Dev: {len(dev_ids)} conversations')
print(f'Val: {len(val_ids)} conversations')
print(f'Eval: {len(evl_ids)} conversations')
print(f'\nDev-Val overlap: {dev_ids & val_ids}')
print(f'Dev-Eval overlap: {dev_ids & evl_ids}')
print(f'Val-Eval overlap: {val_ids & evl_ids}')
print(f'\nLeak check: {"PASS" if not (dev_ids & val_ids) and not (dev_ids & evl_ids) and not (val_ids & evl_ids) else "FAIL"}')